In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import joblib


In [5]:
df = pd.read_csv('heart.csv')
print(df.shape)
print(df['target'].value_counts())
print(df.isnull().sum().sort_values(ascending=False))

(1025, 14)
target
1    526
0    499
Name: count, dtype: int64
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64


In [6]:
X = df.drop('target', axis=1)
y = df['target']

In [8]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (820, 13)
Test: (205, 13)


In [11]:
# Logistic Regression
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)
print(classification_report(y_test, model.predict(X_test)))

              precision    recall  f1-score   support

           0       0.89      0.70      0.78       100
           1       0.76      0.91      0.83       105

    accuracy                           0.81       205
   macro avg       0.82      0.81      0.81       205
weighted avg       0.82      0.81      0.81       205



In [12]:
# Random Forest
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
print(classification_report(y_test, model.predict(X_test)))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       100
           1       1.00      1.00      1.00       105

    accuracy                           1.00       205
   macro avg       1.00      1.00      1.00       205
weighted avg       1.00      1.00      1.00       205



In [13]:
# XGBoost
model = XGBClassifier(random_state=42, eval_metric='logloss')
model.fit(X_train, y_train)

print(classification_report(y_test, model.predict(X_test)))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       100
           1       1.00      1.00      1.00       105

    accuracy                           1.00       205
   macro avg       1.00      1.00      1.00       205
weighted avg       1.00      1.00      1.00       205



In [17]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

params = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'min_samples_leaf': [2, 4]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    params,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)
print("Best params:", grid.best_params_)
print("Best F1:", grid.best_score_)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best params: {'max_depth': 5, 'min_samples_leaf': 2, 'n_estimators': 200}
Best F1: 0.9097636862631806


In [19]:
best_rf = RandomForestClassifier(**grid.best_params_, random_state=42)
best_rf.fit(X_train, y_train)
print(classification_report(y_test, best_rf.predict(X_test)))


              precision    recall  f1-score   support

           0       0.97      0.91      0.94       100
           1       0.92      0.97      0.94       105

    accuracy                           0.94       205
   macro avg       0.94      0.94      0.94       205
weighted avg       0.94      0.94      0.94       205



In [16]:
joblib.dump(best_rf, 'heart_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
print("Model saved")

Model saved
